In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
import searoute as sr


In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "prepare_chokepoints",
)

### File paths

In [ ]:
trade_options_fn = snakemake.input.trade_options
bus_locations_fn = snakemake.input.bus_locations

# Output
trade_options_chokepoints_fn = snakemake.output.trade_options_chokepoints

# Port selection mode: "closest" (region centroid → nearest sea node)
#                  or "predefined" (hand-curated city coordinates from the notes column)
port_mode = snakemake.config.get("trade", {}).get("shipping_routes", {}).get("ports", "closest")
print(f"Port selection mode: '{port_mode}'")


### Load data

In [ ]:
trade_options = pd.read_csv(trade_options_fn, header=0)
bus_locations = pd.read_csv(bus_locations_fn, header=0)

print(f"Trade options: {len(trade_options)} routes")
print(f"Bus locations: {len(bus_locations)} regions")

# Fallback / "closest" mode: one centroid coordinate per region
region_coords = {
    row["region_name"]: [row["long"], row["lat"]]
    for _, row in bus_locations.iterrows()
}


In [ ]:
# Hand-curated city coordinates used in "predefined" port mode.
# Keys match city names (lower-cased) that appear in the "notes" column of trade_opt.csv.
CITY_COORDS = {
    "los angeles":       [-118.24, 33.99],
    "la":                [-118.24, 33.99],
    "hong kong":         [ 114.16, 22.28],
    "hong jong":         [ 114.16, 22.28],  # typo in notes
    "santiago":          [ -70.65,-33.46],
    "lima":              [ -77.04,-12.04],
    "veracruz":          [ -96.13, 19.19],
    "veracruz (mx)":     [ -96.13, 19.19],
    "veracrus":          [ -96.13, 19.19],  # typo in notes
    "veracrus (mx)":     [ -96.13, 19.19],
    "bombay":            [  72.84, 18.96],
    "bombay (in)":       [  72.84, 18.96],
    "vladivostok":       [ 131.88, 43.12],
    "porto":             [  -8.63, 41.16],
    "fukuoka":           [ 130.39, 33.57],
    "fukuoka (jp)":      [ 130.39, 33.57],
    "tokyo":             [ 139.69, 35.68],
    "busan":             [ 129.04, 35.10],
    "buenos aires":      [ -58.38,-34.60],
    "buenos aires (arg)":[ -58.38,-34.60],
    "jakarta":           [ 106.84, -6.21],
    "genova":            [   8.93, 44.41],
    "perth":             [ 115.86,-31.95],
    "perth (au)":        [ 115.86,-31.95],
    "cape town":         [  18.42,-33.92],
    "capetown":          [  18.42,-33.92],
    "new york":          [ -73.94, 40.65],
    "muskat":            [  58.59, 23.59],
    "jedda":             [  39.18, 21.49],
    "jedda (sa)":        [  39.18, 21.49],
    "jedda sa":          [  39.18, 21.49],
    "lazaro carderas":   [-102.18, 17.95],
    "manila":            [ 120.98, 14.60],
    "zhuhai":            [ 113.57, 22.27],
    "zhuhai (cn)":       [ 113.57, 22.27],
    "suez":              [  32.55, 29.97],
    "brisbane":          [ 153.03,-27.47],
    "medellin":          [ -75.57,  6.24],
    "medellin (co)":     [ -75.57,  6.24],
    "novorossisjsk":     [  37.77, 44.72],
    "novorossijsk (ru)": [  37.77, 44.72],
    "port kelang":       [ 101.39,  3.01],
    "port kelang (my)":  [ 101.39,  3.01],
    "poert kelang (my)": [ 101.39,  3.01],  # typo in notes
    "port klang":        [ 101.39,  3.01],
    "rotterdam":         [   4.48, 51.91],
    "lagos":             [   3.38,  6.45],
    "tripoli":           [  13.18, 32.90],
    "casablanca":        [  -7.59, 33.59],
    "london":            [  -0.13, 51.51],
    "st.petersburg":     [  30.32, 59.94],
    "miami":             [ -80.19, 25.77],
    "belem":             [ -48.48, -1.46],
    "belem (bra)":       [ -48.48, -1.46],
    "sao paulo":         [ -46.63,-23.55],
    "turbo":             [ -76.73,  8.09],
    "turbo (co)":        [ -76.73,  8.09],
    "gdansk":            [  18.65, 54.35],
    "bilbao":            [  -2.92, 43.26],
    "lisbon":            [  -9.14, 38.72],
    "lisboa":            [  -9.14, 38.72],
    "chennai":           [  80.27, 13.08],
    "tel aviv":          [  34.78, 32.08],
    "geneve (it)":       [   8.93, 44.41],  # Genova written in Italian
    "alger":             [   3.06, 36.73],
    "kairo":             [  31.24, 30.06],
}


def _parse_shipping_coords(notes_str):
    """Parse 'shipping from X to Y' from notes; return ([lon,lat], [lon,lat]) or (None, None)."""
    if not isinstance(notes_str, str):
        return None, None
    m = re.search(r"shipping from (.+?) to (.+?)(?:,|$)", notes_str, re.IGNORECASE)
    if not m:
        return None, None
    c_from = CITY_COORDS.get(m.group(1).strip().lower())
    c_to   = CITY_COORDS.get(m.group(2).strip().lower())
    return c_from, c_to


def get_route_coords(r_from, r_to, notes):
    """
    Return (origin, destination) coordinates for searoute based on port_mode:
      'closest'    – region centroid; searoute snaps to the nearest sea node.
      'predefined' – hand-curated city coordinates parsed from the notes column;
                     falls back to the region centroid when a city is not found.
    """
    if port_mode == "predefined":
        c_from, c_to = _parse_shipping_coords(notes)
        if c_from is None:
            c_from = region_coords.get(r_from)
        if c_to is None:
            c_to = region_coords.get(r_to)
    else:  # "closest"
        c_from = region_coords.get(r_from)
        c_to   = region_coords.get(r_to)
    return c_from, c_to


# Diagnostic: report any routes that still rely on a region centroid
if port_mode == "predefined":
    fallbacks = []
    for _, row in trade_options[trade_options["shipping"] == 1].iterrows():
        c_from, c_to = _parse_shipping_coords(row.get("notes", ""))
        missing = []
        if c_from is None:
            missing.append(f"origin (region: {row['region_from']})")
        if c_to is None:
            missing.append(f"dest (region: {row['region_to']})")
        if missing:
            fallbacks.append(f"  {row['region_from']} -> {row['region_to']}: {', '.join(missing)}  |  notes: '{row.get('notes','')}'")
    if fallbacks:
        print(f"⚠  {len(fallbacks)} route(s) fall back to region centroid:")
        print("\n".join(fallbacks))
    else:
        n_shipping = (trade_options["shipping"] == 1).sum()
        print(f"✓ All {n_shipping} shipping routes resolved via predefined city coordinates.")
else:
    print(f"Using region centroids for all {(trade_options['shipping']==1).sum()} shipping routes.")


### Compute chokepoints per route using `searoute`

For each shipping route in `trade_opt.csv`, compute the shortest sea route and record which passages (chokepoints) are traversed.

In [ ]:
# Chokepoints to consider from config (mga-chokepoints threshold_value)
config_chokepoints = set(
    snakemake.config.get("scenario", {})
    .get("mga-chokepoints", {})
    .get("modifiers", {})
    .get("mga", {})
    .get("threshold_value", [])
)
print(f"Chokepoints to track (from config): {sorted(config_chokepoints)}")

chokepoints_list = []

for idx, row in trade_options.iterrows():
    r_from = row["region_from"]
    r_to   = row["region_to"]

    # Only compute for shipping routes
    if row["shipping"] != 1:
        chokepoints_list.append("")
        continue

    origin, destination = get_route_coords(r_from, r_to, row.get("notes", ""))

    if origin is None or destination is None:
        print(f"  Warning: missing coordinates for {r_from} -> {r_to}")
        chokepoints_list.append("")
        continue

    try:
        route = sr.searoute(
            origin, destination,
            restrictions=[],  # allow all passages so we can detect them
            return_passages=True,
        )
        passages = route["properties"].get("traversed_passages", [])
        # Filter to only chokepoints listed in config
        relevant = [p for p in passages if p in config_chokepoints]
        chokepoints_list.append(";".join(relevant) if relevant else "")
    except Exception as e:
        print(f"  Error computing route {r_from} -> {r_to}: {e}")
        chokepoints_list.append("")

trade_options["chokepoints"] = chokepoints_list

print(f"\nRoutes with config chokepoints: {(trade_options['chokepoints'] != '').sum()} / {len(trade_options)}")


In [ ]:
# Display routes and their chokepoints
display_cols = ["region_from", "region_to", "chokepoints"]
routes_with_cp = trade_options[trade_options["chokepoints"] != ""][display_cols]
print(f"Routes passing through chokepoints ({len(routes_with_cp)}):")
routes_with_cp

### Save enriched trade options

In [ ]:
trade_options.to_csv(trade_options_chokepoints_fn, index=False)
print(f"Saved to {trade_options_chokepoints_fn}")

### Map: Shipping routes and chokepoints

Global map of all shipping routes, colored by whether they pass through chokepoints. Chokepoint locations are marked with red stars.

In [ ]:
# Known chokepoint coordinates (lon, lat) — only for config chokepoints
all_chokepoint_coords = {
    "babalmandab": [ 43.3, 12.6],
    "bosporus":    [ 29.0, 41.1],
    "gibraltar":   [ -5.6, 36.0],
    "suez":        [ 32.3, 30.5],
    "panama":      [-79.5,  9.1],
    "ormuz":       [ 56.3, 26.6],
    "northwest":   [-95.0, 74.0],
    "malacca":     [101.0,  2.5],
    "sunda":       [105.8, -6.1],
    "chili":       [-75.0,-55.0],
    "south_africa":[ 20.0,-34.8],
    "bering":      [-169.0,65.5],
}
chokepoint_coords = {k: v for k, v in all_chokepoint_coords.items() if k in config_chokepoints}

print(f"Plotting {len(chokepoint_coords)} config chokepoints: {sorted(chokepoint_coords.keys())}")

# Compute sea routes for all shipping pairs (store geometries for plotting)
routes_geo = []
for idx, row in trade_options.iterrows():
    if row["shipping"] != 1:
        continue
    r_from = row["region_from"]
    r_to   = row["region_to"]
    origin, destination = get_route_coords(r_from, r_to, row.get("notes", ""))
    if origin is None or destination is None:
        continue
    try:
        route = sr.searoute(origin, destination, restrictions=[], return_passages=True)
        cp_str = row.get("chokepoints", "")
        has_cp = pd.notna(cp_str) and cp_str != ""
        routes_geo.append({
            "coords": route["geometry"]["coordinates"],
            "has_chokepoint": has_cp,
            "chokepoints": cp_str if has_cp else "",
            "from": r_from,
            "to": r_to,
        })
    except Exception:
        pass

print(f"Computed {len(routes_geo)} route geometries")


In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.Robinson())

# Map features
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", edgecolor="none")
ax.add_feature(cfeature.OCEAN, facecolor="#dbe9f4")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color="grey")
ax.add_feature(cfeature.BORDERS, linewidth=0.2, color="lightgrey")
ax.set_global()

# Plot routes without chokepoints first (background, grey)
for r in routes_geo:
    if not r["has_chokepoint"]:
        coords = np.array(r["coords"])
        ax.plot(coords[:, 0], coords[:, 1],
                color="#000000", linewidth=0.4, alpha=0.5,
                transform=ccrs.Geodetic())

# Plot routes with chokepoints on top (colored)
for r in routes_geo:
    if r["has_chokepoint"]:
        coords = np.array(r["coords"])
        ax.plot(coords[:, 0], coords[:, 1],
                color="#d62728", linewidth=0.6, alpha=0.35,
                transform=ccrs.Geodetic())

# Plot region locations
for name, (lon, lat) in region_coords.items():
    ax.plot(lon, lat, marker="o", color="#1f77b4", markersize=5,
            transform=ccrs.PlateCarree(), zorder=10)
    ax.text(lon + 2, lat + 2, name.replace("_", " "), fontsize=5,
            transform=ccrs.PlateCarree(), zorder=10,
            ha="left", va="bottom", color="#333333")

# Plot chokepoint locations
for cp_name, (lon, lat) in chokepoint_coords.items():
    ax.plot(lon, lat, marker="*", color="#d62728", markersize=12,
            markeredgecolor="white", markeredgewidth=0.5,
            transform=ccrs.PlateCarree(), zorder=20)
    label = cp_name.replace("_", " ").title()
    ax.text(lon + 3, lat - 3, label, fontsize=6, fontweight="bold",
            transform=ccrs.PlateCarree(), zorder=20,
            ha="left", va="top", color="#d62728",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.7, edgecolor="none"))

legend_elements = [
    Line2D([0], [0], color="#d62728", linewidth=1.5, alpha=0.6, label="Route through chokepoint"),
    Line2D([0], [0], color="#000000", linewidth=1.5, alpha=0.8, label="Route without chokepoint"),
    Line2D([0], [0], marker="*", color="#d62728", linestyle="None",
           markersize=12, markeredgecolor="white", label="Chokepoint"),
    Line2D([0], [0], marker="o", color="#1f77b4", linestyle="None",
           markersize=6, label="Region"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=8, framealpha=0.9)

ax.set_title("Global Shipping Routes and Maritime Chokepoints")

plt.tight_layout()
plt.savefig(snakemake.output.map_chokepoints, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.map_chokepoints_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"\nRoutes through chokepoints: {sum(1 for r in routes_geo if r['has_chokepoint'])} / {len(routes_geo)}")
